# Adult: Sensitive Balancing Preprocessing

Author: Ilse Harmers \
Last modified: February 23, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import utils

## Preprocessing Adult Dataset

In [ ]:
# Importing Adult dataset (downloaded from https://archive.ics.uci.edu/dataset/2/adult on May 12, 2025).

# Assigning file paths and column names for importing the dataset. 
train_path = "../adult/adult.data"
test_path = "../adult/adult.test"
column_names = ["age", "workclass", "fnlweight", "education", "education-num", "marital-status", "occupation", 
                "relationship", "race", "sex",  "capital-gain", "capital-loss", "hours-per-week", "native-country", 
                "income"]

# Importing train and test sets as pandas DataFrames.
adult_train = pd.read_csv(train_path, names = column_names)
adult_test = pd.read_csv(test_path, names = column_names)

# Concatenating the train and test sets. 
adult = pd.concat([adult_train.reset_index(drop = True), adult_test.reset_index(drop = True)], axis = 0).reset_index(drop = True)
dataset_name = "Adult"
adult

In [ ]:
# Replacing the question marks in the dataset with NaN values.
adult.replace(" ?", np.nan, inplace = True)

# Replacing " >50K." and " <=50K." with " >50K" and " <=50K", respectively. 
adult.replace(" >50K.", " >50K", inplace = True)
adult.replace(" <=50K.", " <=50K", inplace = True)

In [ ]:
# Print names of columns in the dataset as a check.
print(f"{dataset_name} contains {len(column_names)} columns: \n {column_names} \n")

# Check dtypes of dataset's columns.
print("Columns' dtypes:\n", adult.dtypes)
print()

# Count missing values (reported as NaNs) in each column.
missing_values = np.array([adult[col].isna().sum() for col in column_names])

# This if-elif statement checks whether the columns in the dataset contain missing values that are reported as NaNs. 
if missing_values.any() == False:   # no non-zero entries
    print(f"{dataset_name} has no missing values in any of its columns.")
elif missing_values.any() == True:   # at least one non-zero entry (indicating the presence of missing values)
    print(f"There are {np.sum(missing_values)} missing values in {dataset_name}'s columns.")   
    for i in range(len(missing_values)):
        print(f"\t{column_names[i]}: {missing_values[i]} values")

In [ ]:
# Statistical summary of the dataset.
adult.describe()

### Removing Missing Values & Dropping Columns

In [ ]:
# Removing rows with missing values. 
adult.dropna(inplace=True)

# Count missing values (reported as NaNs) in each column.
missing_values = np.array([adult[col].isna().sum() for col in column_names])

# This if-elif statement checks whether the columns in the dataset contain missing values that are reported as NaNs. 
if missing_values.any() == False:   # no non-zero entries
    print(f"{dataset_name} has no missing values in any of its columns.")
elif missing_values.any() == True:   # at least one non-zero entry (indicating the presence of missing values)
    print(f"There are {np.sum(missing_values)} missing values in {dataset_name}'s columns.")   
    for i in range(len(missing_values)):
        print(f"\t{column_names[i]}: {missing_values[i]} values")

In [ ]:
# See Fabris et al. (2022) about 'education' and 'fnlweight' considerations.
adult = adult.drop(columns=["education", "fnlweight"])

## Sensitive Balancing Preprocessing

In [ ]:
# Splitting the Adult dataset into a train and test set with a 80:20 ratio.
adult_train, adult_test = train_test_split(adult, train_size = 0.80, random_state = 42)

In [ ]:
def sens_preprocessing(df):
    """This function carries out the sensitive balancing preprocessing by randomly undersampling the demographic groups such that each group 
    has the same size. The demographic groups for Adult are Male-<=50K (M0), Male->50K (M1), Female-<=50K (F0) and Female->50K (F1). For Adult, F1 is
    the smallest group. So, M0, M1 and F0 will be undersampled in order to have the same size as F1."""
    
    females = df[df["sex"] == " Female"].shape[0]
    print("females: ", females) 
    males = df[df["sex"] == " Male"].shape[0]
    print("males: ", males) 

    # Creating a new column in the DataFrame with the demographic group labels. 
    df.loc[df.loc[(df["sex"] == " Male") & (df["income"] == " <=50K")].index, "sex-income"] = "M0"
    df.loc[df.loc[(df["sex"] == " Male") & (df["income"] == " >50K")].index, "sex-income"] = "M1"
    df.loc[df.loc[(df["sex"] == " Female") & (df["income"] == " <=50K")].index, "sex-income"] = "F0"
    df.loc[df.loc[(df["sex"] == " Female") & (df["income"] == " >50K")].index, "sex-income"] = "F1"
    f1_rows = df[df["sex-income"] == "F1"].shape[0]
    print("Rows of F1: ", f1_rows)

    # Plotting the label distribution of the demographic groups before preprocessing.
    utils.countplot(data = df, column_name = "sex-income", order = ["F0", "F1", "M0", "M1"], fig_size = (8, 5))

    # Undersampling the demographic groups such that all demographic groups have the same size.
    df_bal = pd.concat([df[df["sex-income"] == "M0"].sample(f1_rows, random_state = 42),
                        df[df["sex-income"] == "M1"].sample(f1_rows, random_state = 42),
                        df[df["sex-income"] == "F0"].sample(f1_rows, random_state = 42),
                        df[df["sex-income"] == "F1"]], axis = 0)

    # Plotting the label distribution of the demographic groups after preprocessing.
    utils.countplot(data = df_bal, column_name = "sex-income", order = ["F0", "F1", "M0", "M1"], fig_size = (6, 5))

    # Resetting the DataFrame.
    df_bal = df_bal.drop(columns = ["sex-income"])
    df = df.drop(columns = ["sex-income"])

    return df_bal

In [ ]:
# Preprocessing the train and test set.
train_bal = sens_preprocessing(adult_train)
test_bal = sens_preprocessing(adult_test)

In [ ]:
# Statistical summary of the train set.
print(train_bal.describe())

# Saving train and test splits to csv files.
#test_bal.to_csv("./train-test-datasets/adult/adult_test.csv", index = False)
#train_bal.to_csv("./train-test-datasets/adult/adult_train.csv", index = False)

In [ ]:
# Computing the demographic parity and disparate impact metrics for the train and test 'balanced' datasets.
df = train_bal #test_bal 
# Encoding the sensitive and target attributes for the fairness analysis.
bal_encoded = utils.one_hot_encode(df[["sex", "income"]], order = [[" <=50K", " >50K"], [" Female", " Male"]])

print("Demographic parity:", utils.demographic_parity(df = bal_encoded, s = "sex", y = "income"))
print("Disparate impact:", utils.disparate_impact(df = bal_encoded, s = "sex", y = "income"))